In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 25 — Frame-component differential geometry (`jacopy.frame_calc`)

Companion notebook to [25_frame_calc.md](25_frame_calc.md). `jacopy.frame_calc` is jacopy's **component-level submodule** for concrete metric calculations: given a metric `g` on a frame, compute Christoffel symbols, Riemann curvature, Ricci tensor, scalar curvature, Einstein tensor — with step-by-step derivation transcripts that bridge to `ProofChain` for paper-grade LaTeX output.

Requires SymPy: `pip install "jacopy[components]"`.

## Quick taste — Schwarzschild vacuum in five lines

In [ ]:
from jacopy.frame_calc import einstein_tensor, levi_civita
from jacopy.frame_calc.library import schwarzschild

F, g = schwarzschild()
G = einstein_tensor(levi_civita(g), g)
print(f'G.is_vacuum() = {G.is_vacuum()}')

## Frame setup — `CoordinateFrame`

Most physics literature uses coordinate frames (`e_a = ∂/∂x^a`). The frame's `derivative(f, a)` is `∂f/∂x^a`; `gamma(a, b, c) = 0` (coordinate frames are holonomic).

In [ ]:
from jacopy.frame_calc import CoordinateFrame
import sympy as sp

t, r, theta, phi = sp.symbols('t r theta phi')
F = CoordinateFrame([t, r, theta, phi])
print(F)
print('dim:', F.dim)
print('e_r(r²) =', F.derivative(r**2, 1))
print('γ^a_bc =', F.gamma(0, 1, 0))

## `ComponentMetric` and `inverse()`

Symmetry checked at construction. `inverse()` returns `g^{ab}` as a `(2, 0)` tensor.

In [ ]:
from jacopy.frame_calc import ComponentMetric

M = sp.Symbol('M', positive=True)
g = ComponentMetric(F, sp.Matrix([
    [-(1 - 2*M/r),   0,                0,    0],
    [0,              1/(1 - 2*M/r),    0,    0],
    [0,              0,                r**2, 0],
    [0,              0,                0,    r**2 * sp.sin(theta)**2],
]))
print('g[0,0] =', g[0, 0])
print('g_inv[0,0] =', g.inverse()[0, 0])
# Verify g^{ac} g_{cb} = δ^a_b for one entry
g_inv = g.inverse()
delta_00 = sp.simplify(sum(g_inv[0, c] * g[c, 0] for c in range(4)))
print('δ^0_0 =', delta_00)

## Levi-Civita Christoffel symbols via Koszul formula

The unique torsion-free metric-compatible connection.

In [ ]:
from jacopy.frame_calc import levi_civita

LC = levi_civita(g)
print(f'# non-zero Christoffel: {len(LC.nonzero_components())}')
print(f'Γ^t_tr = {LC[0, 0, 1]}')
print(f'Γ^θ_rθ = {LC[2, 1, 2]}')
print(f'Γ^r_θθ = {LC[1, 2, 2]}')

## Step-by-step derivation transcript

Each Christoffel computation records `KoszulStep`s. Use `format_derivation` for plain text or `derivation_chain` for `ProofChain` → LaTeX.

In [ ]:
print(LC.format_derivation(0, 0, 1))

## Curvature, Ricci, Einstein

Schwarzschild is Ricci-flat — `Ric = 0`, `R = 0`, `G = 0`. This is the vacuum field equation result.

In [ ]:
from jacopy.frame_calc import (
    curvature, ricci, ricci_scalar, einstein_tensor,
)

R = curvature(LC)
Ric = ricci(LC)
R_scalar = ricci_scalar(LC, g)
G = einstein_tensor(LC, g)

print(f'curvature.is_zero():  {R.is_zero()}  (NOT flat)')
print(f'Ric.is_zero():        {Ric.is_zero()}')
print(f'R_scalar:             {R_scalar}')
print(f'G.is_vacuum():        {G.is_vacuum()}')

## Optimised mode for Kerr-class metrics

Default mode runs `sympy.simplify` on every Christoffel / Ricci / curvature entry. For Kerr (off-diagonal + complex denominators), this blows up. **Optimised mode** skips per-entry simplify; expressions stay raw but mathematically correct. Trade-off: no derivation traces in optimised mode.

In [ ]:
from jacopy.frame_calc.library import kerr
import time

F_kerr, g_kerr = kerr()
t0 = time.perf_counter()
G_kerr = einstein_tensor(
    levi_civita(g_kerr, optimized=True), g_kerr, optimized=True
)
elapsed = time.perf_counter() - t0
print(f'Kerr full pipeline: {elapsed:.1f} s')
print(f'G.is_vacuum(): {G_kerr.is_vacuum()}')

## Library fixtures

Ready-made factories: `minkowski`, `schwarzschild`, `frw`, `kerr`. Each accepts `Symbol` / `Function` overrides.

In [ ]:
from jacopy.frame_calc.library import minkowski, frw

# Minkowski 4D — flat
F_m, g_m = minkowski()
G_m = einstein_tensor(levi_civita(g_m), g_m)
print(f'Minkowski G.is_vacuum(): {G_m.is_vacuum()}')

# FRW (k=0, a(t) symbolic) — non-vacuum cosmology
F_frw, g_frw = frw()
G_frw = einstein_tensor(levi_civita(g_frw), g_frw)
print(f'FRW G.is_zero(): {G_frw.is_zero()} (Friedmann eq form)')
print(f'FRW G[0,0] = {sp.simplify(G_frw[0, 0])}')

## ProofChain bridge — paper-grade LaTeX

Each tracked tensor's `derivation_chain(...)` returns a `ProofChain` compatible with `chain_to_latex_document`. The `SymPyAtom` wrapper bridges SymPy expressions into jacopy's `Expr` for ProofStep storage.

In [ ]:
from jacopy.display import chain_to_latex

chain = LC.derivation_chain(0, 0, 1)   # Γ^t_tr
print(f'chain length: {len(chain.steps)}')
print(f'first step rule: {chain.steps[0].rule}')
print(f'first step tag: {chain.steps[0].provenance_tag}')

## Summary

* `jacopy.frame_calc` is the component-level submodule for concrete metric calculations.
* Three frame types (CoordinateFrame, Tetrad, AbstractFrame) share a common `Frame` protocol; higher-level operations are frame-agnostic.
* Pipeline: `g → LC → R → Ric → R_scalar → G`. Default mode records full derivation traces; optimised mode skips them for Kerr-class performance.
* Library fixtures cover standard metrics; users build custom metrics on any frame.
* `derivation_chain(...)` lifts any per-entry trace to a `ProofChain` for paper-grade LaTeX rendering.
* SymPy is an opt-in dependency under `[components]`.